# FraudGraph — Stage 2: Heterogeneous Graph Construction

Turn the flat 590k-row transaction table into a **HeteroData** graph with four
node types and four edge types. This is the structural artifact GraphSAGE
consumes in Stage 3. Transaction nodes carry the *same* full feature matrix
used by `xgboost_full` — so the Stage 3 comparison isolates graph structure
as the only difference.

## 0. Setup

In [1]:
!pip install -q torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.5 MB/s eta 0:00:0000:01


In [2]:
import os, pickle, numpy as np, pandas as pd, torch
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import HeteroData

DATA = "/kaggle/input/competitions/ieee-fraud-detection"
OUT = "/kaggle/working"
print("files:", os.listdir(DATA))

files: ['sample_submission.csv', 'test_identity.csv', 'train_identity.csv', 'test_transaction.csv', 'train_transaction.csv']


## 1. Reload & re-run Stage 1 preprocessing

In [3]:
txn = pd.read_csv(f"{DATA}/train_transaction.csv")
idn = pd.read_csv(f"{DATA}/train_identity.csv")
df = txn.merge(idn, on="TransactionID", how="left")
df = df.sort_values("TransactionDT").reset_index(drop=True)
cutoff = df["TransactionDT"].quantile(0.80)
is_train = df["TransactionDT"] <= cutoff
print(f"rows: {len(df):,}  cutoff: {int(cutoff):,}  train={is_train.sum():,} test={(~is_train).sum():,}")

rows: 590,540  cutoff: 12,192,853  train=472,432 test=118,108


In [4]:
df["card1"] = df["card1"].fillna(-999)
df["addr1"] = df["addr1"].fillna(-999)
df["dt"] = pd.to_datetime(df["TransactionDT"], unit="s")
df["hour"] = ((df["TransactionDT"] // 3600) % 24).astype(int)
df["cm_freq"] = df.groupby(["card1", "ProductCD", "addr1"]).cumcount()

g = df.groupby("card1")["TransactionAmt"]
mean_prev = g.transform(lambda s: s.expanding().mean().shift())
std_prev = g.transform(lambda s: s.expanding().std().shift())
df["amt_zscore"] = ((df["TransactionAmt"] - mean_prev) / std_prev).replace(
    [np.inf, -np.inf], np.nan).fillna(0)

df = df.sort_values(["card1", "dt"])
roll = df.set_index("dt").groupby("card1")["TransactionAmt"]
df["velocity_1h"] = roll.rolling("1h").count().values
df["velocity_6h"] = roll.rolling("6h").count().values
df["velocity_24h"] = roll.rolling("24h").count().values
df = df.sort_values("TransactionDT").reset_index(drop=True)
is_train = df["TransactionDT"] <= cutoff
print("engineered features ready")

engineered features ready


## 2. Node identity keys

Each node type gets a stable integer id 0..N-1. We build **lookup maps** from
raw key (e.g., a `card1` value) to node index, then apply them vectorized to
build edge tensors.

In [5]:
DEVICE_UNK = "UNKNOWN"
df["DeviceInfo_key"] = df["DeviceInfo"].fillna(DEVICE_UNK).astype(str)
df["merchant_key"] = df["ProductCD"].astype(str) + "|" + df["addr1"].astype(str)

card_ids = pd.Index(sorted(df["card1"].unique()))
merch_ids = pd.Index(sorted(df["merchant_key"].unique()))
dev_ids   = pd.Index(sorted(df["DeviceInfo_key"].unique()))

card_map  = {v: i for i, v in enumerate(card_ids)}
merch_map = {v: i for i, v in enumerate(merch_ids)}
dev_map   = {v: i for i, v in enumerate(dev_ids)}

df["_card_idx"]  = df["card1"].map(card_map).astype("int64")
df["_merch_idx"] = df["merchant_key"].map(merch_map).astype("int64")
df["_dev_idx"]   = df["DeviceInfo_key"].map(dev_map).astype("int64")

print(f"transactions: {len(df):,}")
print(f"cards:        {len(card_map):,}")
print(f"merchants:    {len(merch_map):,}")
print(f"devices:      {len(dev_map):,}  ({DEVICE_UNK}={dev_map[DEVICE_UNK]})")

transactions: 590,540
cards:        13,553
merchants:    532
devices:      1,787  (UNKNOWN=1557)


## 3. Transaction node features (full matrix + standard-scaled)

Same recipe as `xgboost_full`: every usable numeric + label-encoded categorical.
We additionally **StandardScale** (fit on train slice) — trees don't care about
scale, GNNs do.

In [6]:
drop_cols = ["TransactionID", "isFraud", "TransactionDT", "dt",
             "DeviceInfo_key", "merchant_key", "_card_idx", "_merch_idx", "_dev_idx"]
cat_cols = [c for c in df.columns if df[c].dtype == "object" and c not in drop_cols]
num_cols = [c for c in df.columns
            if c not in drop_cols + cat_cols and str(df[c].dtype) != "datetime64[ns]"]

cat_maps = {}
X = df[num_cols + cat_cols].copy()
for c in cat_cols:
    m = {v: i for i, v in enumerate(sorted(X[c].dropna().unique(), key=str))}
    cat_maps[c] = m
    X[c] = X[c].map(m).fillna(-1).astype("int32")
med = X.loc[is_train, num_cols].median()
X[num_cols] = X[num_cols].fillna(med)
feat_cols = num_cols + cat_cols

scaler = StandardScaler()
scaler.fit(X.loc[is_train].values)
X_scaled = scaler.transform(X.values).astype(np.float32)
print(f"transaction node features: {X_scaled.shape}  dtype={X_scaled.dtype}")

transaction node features: (590540, 437)  dtype=float32


## 4. Edge tensors

Three from spec (transaction → outward) and one custom (`velocity_cluster`,
transaction → transaction, same card within 5 min). Edges are causal:
`velocity_cluster` points **earlier → later**, so at inference time a fresh
transaction receives messages from its recent past on the same card and never
from the future.

We add reverse edges via `ToUndirected` in Stage 3 for message passing.

In [7]:
tx_idx = np.arange(len(df), dtype=np.int64)

e_uses_card    = torch.tensor(np.stack([tx_idx, df["_card_idx"].values]), dtype=torch.long)
e_at_merchant  = torch.tensor(np.stack([tx_idx, df["_merch_idx"].values]), dtype=torch.long)

has_id = df["DeviceInfo"].notna().values
e_on_device = torch.tensor(
    np.stack([tx_idx[has_id], df.loc[has_id, "_dev_idx"].values]), dtype=torch.long)

print("uses_card    edges:", e_uses_card.shape[1])
print("at_merchant  edges:", e_at_merchant.shape[1])
print("on_device    edges:", e_on_device.shape[1], f"({has_id.mean():.1%} of tx have identity)")

uses_card    edges: 590540
at_merchant  edges: 590540
on_device    edges: 118666 (20.1% of tx have identity)


### velocity_cluster: same card + within 5-minute window

We sort by (card1, time), then for each transaction find all subsequent
transactions on the same card whose `TransactionDT` is within +300 seconds.
`np.searchsorted` makes this O(N log N) instead of O(N²).

In [8]:
WIN = 300  # seconds
order = np.argsort(df["TransactionDT"].values, kind="stable")
src_all, dst_all = [], []

for card_val, group_pos in df.groupby("card1").indices.items():
    if len(group_pos) < 2:
        continue
    group_pos = np.asarray(group_pos)
    times = df["TransactionDT"].values[group_pos]
    order_local = np.argsort(times, kind="stable")
    times = times[order_local]
    ids = group_pos[order_local]
    upper = np.searchsorted(times, times + WIN, side="right")
    for i in range(len(ids)):
        j0, j1 = i + 1, upper[i]
        if j1 > j0:
            src_all.append(np.full(j1 - j0, ids[i], dtype=np.int64))
            dst_all.append(ids[j0:j1])

if src_all:
    src_vc = np.concatenate(src_all)
    dst_vc = np.concatenate(dst_all)
    e_velocity_cluster = torch.tensor(np.stack([src_vc, dst_vc]), dtype=torch.long)
else:
    e_velocity_cluster = torch.zeros((2, 0), dtype=torch.long)

print("velocity_cluster edges:", e_velocity_cluster.shape[1])
assert e_velocity_cluster.shape[1] > 0, "velocity_cluster is empty — check window/index alignment"

velocity_cluster edges: 114439


### velocity_cluster: is it capturing fraud rings?

Quick sanity check: what fraction of the endpoints of these edges are labeled fraud?
If fraud is 3.5% overall but >10% inside velocity clusters, this edge type is
carrying real signal — exactly the coordinated-fraud pattern trees can't see.

In [9]:
y = df["isFraud"].values
endpoint_fraud = y[np.unique(np.concatenate([e_velocity_cluster[0].numpy(),
                                              e_velocity_cluster[1].numpy()]))].mean()
print(f"Fraud rate overall:                {y.mean():.3%}")
print(f"Fraud rate on velocity_cluster tx: {endpoint_fraud:.3%}")
print(f"Lift: {endpoint_fraud / y.mean():.2f}×")

Fraud rate overall:                3.499%
Fraud rate on velocity_cluster tx: 5.637%
Lift: 1.61×


## 5. Assemble HeteroData

In [10]:
data = HeteroData()
data["transaction"].x = torch.from_numpy(X_scaled)
data["transaction"].y = torch.from_numpy(y.astype(np.int64))
data["transaction"].train_mask = torch.from_numpy(is_train.values)
data["transaction"].test_mask  = torch.from_numpy((~is_train).values)

data["card"].num_nodes     = len(card_map)
data["merchant"].num_nodes = len(merch_map)
data["device"].num_nodes   = len(dev_map)

data["transaction", "uses_card",         "card"     ].edge_index = e_uses_card
data["transaction", "at_merchant",       "merchant" ].edge_index = e_at_merchant
data["transaction", "on_device",         "device"   ].edge_index = e_on_device
data["transaction", "velocity_cluster",  "transaction"].edge_index = e_velocity_cluster

print(data)

HeteroData(
  transaction={
    x=[590540, 437],
    y=[590540],
    train_mask=[590540],
    test_mask=[590540],
  },
  card={ num_nodes=13553 },
  merchant={ num_nodes=532 },
  device={ num_nodes=1787 },
  (transaction, uses_card, card)={ edge_index=[2, 590540] },
  (transaction, at_merchant, merchant)={ edge_index=[2, 590540] },
  (transaction, on_device, device)={ edge_index=[2, 118666] },
  (transaction, velocity_cluster, transaction)={ edge_index=[2, 114439] }
)


## 6. Validation checks

In [11]:
n_tx = data["transaction"].num_nodes
assert data["transaction"].x.shape == (n_tx, X_scaled.shape[1])
assert data["transaction"].y.shape == (n_tx,)
assert data["transaction"].train_mask.sum().item() + data["transaction"].test_mask.sum().item() == n_tx
assert (data["transaction", "uses_card", "card"].edge_index[0].max() < n_tx)
assert (data["transaction", "uses_card", "card"].edge_index[1].max() < data["card"].num_nodes)
assert (data["transaction", "at_merchant", "merchant"].edge_index[1].max() < data["merchant"].num_nodes)
assert (data["transaction", "on_device", "device"].edge_index[1].max() < data["device"].num_nodes)
assert (data["transaction", "velocity_cluster", "transaction"].edge_index.max() < n_tx)

print(f"Feature dim: {data['transaction'].x.shape[1]}")
print(f"Train fraud: {y[is_train.values].mean():.3%}   Test fraud: {y[(~is_train).values].mean():.3%}")
print("All structural checks passed.")

Feature dim: 437
Train fraud: 3.514%   Test fraud: 3.441%
All structural checks passed.


## 7. Save artifacts

In [12]:
torch.save(data, f"{OUT}/graph.pt")

graph_meta = {
    "num_transactions": n_tx,
    "num_cards": len(card_map),
    "num_merchants": len(merch_map),
    "num_devices": len(dev_map),
    "feature_cols": feat_cols,
    "cat_cols": cat_cols,
    "num_cols": num_cols,
    "cat_maps": cat_maps,
    "medians": med.to_dict(),
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "card_map": card_map,
    "merch_map": merch_map,
    "dev_map": dev_map,
    "split_cutoff": int(cutoff),
    "device_unk": DEVICE_UNK,
    "velocity_window_sec": WIN,
}
with open(f"{OUT}/graph_meta.pkl", "wb") as f:
    pickle.dump(graph_meta, f)

print("Saved: graph.pt, graph_meta.pkl")
print("Download both from the Kaggle Output tab.")

Saved: graph.pt, graph_meta.pkl
Download both from the Kaggle Output tab.
